# SEISMOS quickstart

Requires the core library (`pip install -e ..`) and `matplotlib`.

In [ ]:
import matplotlib.pyplot as plt

from seismos import Encoder, SeismosChunker, signal_diagnostics
from seismos.detectors import second_difference
from seismos.segment import ensure_punkt

ensure_punkt()
encoder = Encoder.from_name("bge")

Three topics, 14 sentences each. With `m` clean boundaries in an otherwise flat
signal of length `N`, the statistic peaks at about `sqrt(N / (6m))`, so at
`k = 2.5` this document is borderline. The last cell shows the effect of `k`.

In [ ]:
document = " ".join([
    'The Atlantic puffin breeds on rocky islands in the North Atlantic.',
    'It nests in burrows dug into cliff-top turf.',
    'Pairs return to the same burrow year after year.',
    'The single egg is incubated by both parents for about six weeks.',
    "Chicks are fed fish carried crosswise in the parent's bill.",
    'A parent can hold more than ten sand eels at once.',
    'Chicks fledge at night to avoid predatory gulls.',
    'Young puffins spend their first years entirely at sea.',
    'They return to land only when ready to breed, at around five years old.',
    'Adults moult their colourful bill plates after the breeding season.',
    'In winter the bill is smaller and duller.',
    'Puffins dive using their wings to swim underwater.',
    'They can reach depths of around sixty metres.',
    'Large colonies are found in Iceland, Norway and Newfoundland.',
    'Central banks set short-term interest rates to manage inflation.',
    'Raising rates makes borrowing more expensive and slows spending.',
    'Lower rates encourage investment but can overheat the economy.',
    'Most central banks target an inflation rate near two percent.',
    'Decisions are announced after scheduled policy meetings.',
    'Markets react within seconds to any surprise in the announcement.',
    'Bond yields often move before the decision is published.',
    'Forward guidance signals where rates are likely to go next.',
    "Quantitative easing expands the central bank's balance sheet.",
    'It involves buying government bonds to push down long-term yields.',
    'Tightening reverses this by letting bonds mature without reinvestment.',
    'Exchange rates respond to changes in the interest rate gap between countries.',
    'A stronger currency makes imports cheaper and lowers inflation.',
    'Independence from government is meant to make policy credible.',
    'Sourdough bread is leavened by a culture of wild yeast and bacteria.',
    'The starter is fed daily with flour and water.',
    'A mature starter doubles in size within a few hours of feeding.',
    'Long fermentation develops the sour flavour and an open crumb.',
    'Bakers mix flour and water first and let it rest before adding salt.',
    'Stretching and folding the dough builds gluten strength.',
    'Bulk fermentation can take four to eight hours at room temperature.',
    'The dough is then shaped into a tight ball or oval.',
    'Many bakers proof the shaped loaf overnight in the refrigerator.',
    'Cold proofing makes the dough easier to score.',
    'A sharp blade cuts a slash that lets the loaf expand in the oven.',
    'Baking in a covered pot traps steam during the first twenty minutes.',
    'Removing the lid lets the crust brown and blister.',
    'The loaf should cool for an hour before slicing.',
])


In [ ]:
chunker = SeismosChunker(encoder, k=2.5, max_tokens=480)
chunks = chunker.chunk(document)
for i, c in enumerate(chunks):
    print(f"[{i}] sentences {c.first_sentence}-{c.last_sentence}: {c.text[:80]}...")

Top: `X[i]`, similarity of sentences `i` and `i+1`. Bottom: `-D[i] / std(D)`
against `k`. Red lines are the boundaries.

In [ ]:
X = chunker.signal(document).X
D = second_difference(X)
splits = [c.last_sentence for c in chunks[:-1]]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)
ax1.plot(range(len(X)), X, marker="o")
ax1.set_ylabel("X[i]")
ax2.bar(range(1, len(X) - 1), -D / D.std())
ax2.axhline(chunker.spec.k, linestyle="--", color="gray")
ax2.set_ylabel("-D[i] / std(D)")
ax2.set_xlabel("i")
for s in splits:
    ax1.axvline(s, color="red", alpha=0.3)
    ax2.axvline(s, color="red", alpha=0.3)
plt.tight_layout()
plt.show()

signal_diagnostics(X)

In [ ]:
for k in (1.5, 2.0, 2.5, 3.0):
    print(k, len(SeismosChunker(encoder, k=k, max_tokens=480).chunk(document)))